In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
data=pd.read_csv("/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Merged files/csv/apple/merged_apple_funadmentals_price_macro.csv")

In [3]:
data

,quarter,period_end,Price,revenue,net_income,ratio assets/libailities,shareholders_equity,gdp_growth,interest_rate
0,Q1 2010,2009-12-26,6.407192,15683000000,3378000000,2.969820,35768000000,0.11,0.120000
1,Q2 2010,2010-03-27,7.127073,13499000000,3074000000,3.221921,39348000000,1.75,0.133333
2,Q3 2010,2010-06-26,7.634115,15700000000,3253000000,2.994587,43111000000,2.91,0.193333
3,Q4 2010,2010-09-25,8.590257,20343000000,4308000000,2.744706,47791000000,3.34,0.186667
4,Q1 2011,2010-12-25,9.775751,26741000000,6004000000,2.704265,54666000000,2.78,0.186667
...,...,...,...,...,...,...,...,...,...
58,Q3 2024,2024-06-29,210.863424,85777000000,21448000000,1.251820,66708000000,3.04,5.330000
59,Q4 2024,2024-09-28,228.456772,94930000000,14736000000,1.184885,56950000000,2.72,5.263333
60,Q1 2025,2024-12-28,248.049447,124300000000,36330000000,1.240719,66758000000,2.53,4.650000
61,Q2 2025,2025-03-29,219.273273,95359000000,24780000000,1.252597,66796000000,1.99,4.330000


In [4]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error



# --- 1) Add lag-1 of Price ---
data["Price_lag1"] = data["Price"].shift(1)

# --- 2) Keep only numeric features (drop non-numeric like 'quarter') ---
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# target and features
y = data["Price"]
X = data[num_cols]  # includes Price_lag1 + fundamentals (all numeric)

# Drop rows with NaNs from lag/cleaning
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

# --- 3) Time-series split (no shuffling) ---
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# --- 4) Pipeline: Standardize -> Ridge (handles multicollinearity) ---
model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0, random_state=42)
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# --- 5) Metrics ---
print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred))



R2: 0.9374214306191058
MAE: 7.2323185288282685
RMSE: 83.73827976850905


In [5]:
# Correlation matrix
corr = data.corr(numeric_only=True)

# Correlation of all features with Price
corr

,Price,revenue,net_income,ratio assets/libailities,shareholders_equity,gdp_growth,interest_rate,Price_lag1
Price,1.000000,0.801229,0.793635,-0.693350,-0.539231,0.140410,0.749632,0.981055
revenue,0.801229,1.000000,0.968697,-0.800725,-0.172506,0.174207,0.570043,0.786312
net_income,0.793635,0.968697,1.000000,-0.696009,-0.235501,0.225333,0.543637,0.778781
ratio assets/libailities,-0.693350,-0.800725,-0.696009,1.000000,0.112150,-0.100184,-0.533222,-0.675092
shareholders_equity,-0.539231,-0.172506,-0.235501,0.112150,1.000000,0.019459,-0.336659,-0.576274
gdp_growth,0.140410,0.174207,0.225333,-0.100184,0.019459,1.000000,0.072416,0.136913
interest_rate,0.749632,0.570043,0.543637,-0.533222,-0.336659,0.072416,1.000000,0.744119
Price_lag1,0.981055,0.786312,0.778781,-0.675092,-0.576274,0.136913,0.744119,1.000000
